In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns

# ====================== 全局统一配置（工程化规范） ======================
# 路径常量
INPUT_PATH = "processed_data/UserBehavior_cleaned.csv"
OUTPUT_DIR = "outputs"
DOCS_DIR = "docs"
REPORT_DIR = "report"

# 绘图全局配置（全平台中文兼容 + 专业风格）
plt.rcParams['font.sans-serif'] = ['SimHei', 'WenQuanYi Micro Hei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.style.use('seaborn-v0_8-whitegrid')
FIG_SIZE_NORMAL = (8, 6)
FIG_SIZE_WIDE = (12, 5)
DPI = 300

# 创建目录
for folder in [OUTPUT_DIR, DOCS_DIR, REPORT_DIR]:
    os.makedirs(folder, exist_ok=True)


# ====================== 工具函数封装 ======================
def stable_quantile_score(series, q=4, reverse=False):
    """稳定分位数打分，避免极值报错"""
    percentiles = np.linspace(0, 100, q + 1)
    bins = np.percentile(series, percentiles)
    bins = np.unique(bins)
    if len(bins) < 2:
        return pd.Series([1] * len(series), index=series.index)
    bins[0] = -np.inf
    bins[-1] = np.inf
    labels_num = pd.cut(series, bins=bins, labels=False, include_lowest=True)
    score = labels_num + 1
    if reverse:
        score = score.max() + 1 - score
    return score.astype(int)


def rfm_8_segment(row):
    """行业标准RFM 8象限用户分层"""
    r, f, m = row['R_score'], row['F_score'], row['M_score']
    if r >= 3 and f >= 3 and m >= 3:
        return "重要价值用户"
    elif r < 3 and f >= 3 and m >= 3:
        return "重要挽留用户"
    elif r >= 3 and f < 3 and m >= 3:
        return "重要深耕用户"
    elif r < 3 and f < 3 and m >= 3:
        return "重要唤回用户"
    elif r >= 3 and f >= 3 and m < 3:
        return "一般价值用户"
    elif r < 3 and f >= 3 and m < 3:
        return "一般挽留用户"
    elif r >= 3 and f < 3 and m < 3:
        return "一般潜力用户"
    else:
        return "低价值流失用户"


def ai_auto_analysis(seg_stats, conv_rate_str, peak_hours_str):
    """AI自动业务洞察生成（项目AI亮点，无第三方接口）"""
    high_pct = seg_stats.iloc[0]['用户占比'] if not seg_stats.empty else 0
    buy_pct = seg_stats.iloc[0]['购买贡献占比'] if not seg_stats.empty else 0
    ai_text = f"""
## AI 智能数据分析洞察
1. 用户结构洞察：仅占{high_pct:.1%}的核心高价值用户，撑起平台{buy_pct:.1%}购买量，符合电商二八定律，运营应重点倾斜核心用户留存。
2. 转化效率洞察：平台整体转化率{conv_rate_str}，若偏低需从商品详情、定价、营销活动三端优化。
3. 时段运营洞察：用户集中活跃在{peak_hours_str}，建议平台广告投放、直播带货、优惠券推送集中在此时间段，提升触达转化率。
4. 商品运营洞察：存在高浏览低转化商品，并非流量不足，而是商品卖点、定价、评价体系存在短板，需单品精细化优化。
"""
    return ai_text


# ====================== 数据加载与预处理 ======================
def load_and_preprocess():
    df = pd.read_csv(INPUT_PATH, parse_dates=['datetime'])
    print(f"✅ 数据加载完成，共 {len(df):,} 行")
    print(f"   时间范围：{df['datetime'].min()} 至 {df['datetime'].max()}")

    df['buy_flag'] = (df['behavior'] == 'buy').astype(np.int8)
    df['pv_flag'] = (df['behavior'] == 'pv').astype(np.int8)
    return df


# ====================== RFM 8象限用户分层分析 ======================
def rfm_analysis(df):
    print("\n--- 1. RFM 8象限 用户分层（行业标准版） ---")
    ref_date = (df['datetime'].max() + pd.Timedelta(days=1)).normalize()
    print(f"   RFM参考日期：{ref_date.date()}")

    # RFM聚合
    rfm = df.groupby('user_id').agg(
        last_time=('datetime', 'max'),
        F=('behavior', 'count'),
        M=('buy_flag', 'sum')
    ).reset_index()
    rfm['R'] = (ref_date - rfm['last_time']).dt.days
    rfm = rfm.drop(columns=['last_time'])

    # 打分
    rfm['R_score'] = stable_quantile_score(rfm['R'], q=4, reverse=True)
    rfm['F_score'] = stable_quantile_score(rfm['F'], q=4, reverse=False)

    rfm['M_score'] = 1
    has_buy = rfm['M'] > 0
    if has_buy.sum() >= 3:
        m_raw = stable_quantile_score(rfm.loc[has_buy, 'M'], q=3, reverse=False)
        rfm.loc[has_buy, 'M_score'] = m_raw + 1
    elif has_buy.sum() > 0:
        rfm.loc[has_buy, 'M_score'] = 2

    # 8象限分层
    rfm['segment'] = rfm.apply(rfm_8_segment, axis=1)
    rfm.to_csv(f"{OUTPUT_DIR}/rfm_segmentation.csv", index=False, encoding='utf-8-sig')
    print(f"✅ RFM8象限分层结果已保存")

    # 分层统计
    segment_stats = rfm.groupby('segment').agg(
        用户数=('user_id', 'count'),
        总购买次数=('M', 'sum')
    ).reset_index()
    segment_stats['用户占比'] = segment_stats['用户数'] / segment_stats['用户数'].sum()
    segment_stats['购买贡献占比'] = segment_stats['总购买次数'] / segment_stats['总购买次数'].sum()

    return rfm, segment_stats


# ====================== 商品品类分析 ======================
def category_item_analysis(df):
    print("\n--- 2. 商品/品类价值分析 ---")
    # 热门品类TOP10
    category_buy = df[df['buy_flag'] == 1].groupby('category_id').size().sort_values(ascending=False).head(
        10).reset_index()
    category_buy.columns = ['品类ID', '购买次数']
    category_buy.to_csv(f"{OUTPUT_DIR}/top_categories.csv", index=False, encoding='utf-8-sig')

    # 商品转化分析
    item_stats = df.groupby('item_id').agg(
        浏览次数=('pv_flag', 'sum'),
        购买次数=('buy_flag', 'sum')
    ).reset_index()
    item_stats['转化率'] = np.where(item_stats['浏览次数'] == 0, 0, item_stats['购买次数'] / item_stats['浏览次数'])
    item_stats['转化率'] = item_stats['转化率'].replace([np.inf, -np.inf], 0)
    problem_items = item_stats[item_stats['浏览次数'] >= 100].nsmallest(20, '转化率').reset_index(drop=True)
    problem_items.to_csv(f"{OUTPUT_DIR}/high_clicks_low_conversion.csv", index=False, encoding='utf-8-sig')

    print("✅ 热门品类、问题商品清单已保存")
    return category_buy, problem_items


# ====================== 可视化绘图模块（全面升级） ======================
def draw_all_plots(segment_stats, rfm, category_buy, problem_items):
    print("\n--- 3. 高级可视化绘图中 ---")
    IMG_DIR = REPORT_DIR

    # 图1：用户分层占比饼图（加数据标签）
    plt.figure(figsize=FIG_SIZE_NORMAL)
    seg_data = segment_stats[['segment', '用户数']].copy()
    wedges, texts, autotexts = plt.pie(
        seg_data['用户数'], labels=seg_data['segment'],
        autopct='%1.1f%%', startangle=90,
        colors=['#e74c3c', '#f39c12', '#3498db', '#2ecc71', '#9b59b6', '#1abc9c', '#95a5a6', '#34495e']
    )
    plt.title('电商用户价值分层占比（RFM8象限）', fontsize=14, pad=20)
    plt.savefig(f"{IMG_DIR}/user_segment_pie.png", dpi=DPI, bbox_inches='tight')
    plt.close()

    # 图2：各分层购买贡献柱状图
    plt.figure(figsize=FIG_SIZE_WIDE)
    bar = sns.barplot(x='segment', y='购买贡献占比', data=segment_stats, palette='Reds_d')
    # 添加数值标签
    for p in bar.patches:
        bar.annotate(f"{p.get_height():.1%}",
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='center', xytext=(0, 5), textcoords='offset points')
    plt.title('各用户分层购买贡献占比', fontsize=14)
    plt.ylabel('购买贡献占比')
    plt.xlabel('用户分层')
    plt.xticks(rotation=45)
    plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.1%}"))
    plt.tight_layout()
    plt.savefig(f"{IMG_DIR}/segment_contribution.png", dpi=DPI, bbox_inches='tight')
    plt.close()

    # 图3：RFM热力图
    plt.figure(figsize=FIG_SIZE_NORMAL)
    rfm_pivot = rfm.pivot_table(index='F_score', columns='R_score', values='user_id', aggfunc='count', fill_value=0)
    sns.heatmap(rfm_pivot, annot=True, fmt='d', cmap='Blues', cbar_kws={'label': '用户数量'})
    plt.title('RFM R-F 分数分布热力图', fontsize=14)
    plt.xlabel('R分数（近度）')
    plt.ylabel('F分数（频次）')
    plt.savefig(f"{IMG_DIR}/rfm_heatmap.png", dpi=DPI, bbox_inches='tight')
    plt.close()

    # 图4：热门品类TOP10
    plt.figure(figsize=FIG_SIZE_WIDE)
    bar2 = sns.barplot(x='品类ID', y='购买次数', data=category_buy, palette='viridis')
    for p in bar2.patches:
        bar2.annotate(f"{int(p.get_height())}",
                      (p.get_x() + p.get_width() / 2., p.get_height()),
                      ha='center', va='center', xytext=(0, 5), textcoords='offset points')
    plt.title('热门品类TOP10 购买次数', fontsize=14)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(f"{IMG_DIR}/top_category_bar.png", dpi=DPI, bbox_inches='tight')
    plt.close()

    # 图5：低转化商品TOP10
    plt.figure(figsize=FIG_SIZE_WIDE)
    problem_items['item_id'] = problem_items['item_id'].astype(str)
    bar3 = sns.barplot(x='item_id', y='转化率', data=problem_items.head(10), palette='coolwarm')
    for p in bar3.patches:
        bar3.annotate(f"{p.get_height():.2%}",
                      (p.get_x() + p.get_width() / 2., p.get_height()),
                      ha='center', va='center', xytext=(0, 5), textcoords='offset points')
    plt.title('高浏览低转化商品TOP10 转化率', fontsize=14)
    plt.xticks(rotation=45)
    plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.2%}"))
    plt.tight_layout()
    plt.savefig(f"{IMG_DIR}/low_conversion_items.png", dpi=DPI, bbox_inches='tight')
    plt.close()

    print("✅ 全部高级可视化图表已保存至 report/")


# ====================== 生成业务报告 + AI洞察 ======================
def generate_business_report(segment_stats):
    print("\n--- 4. 生成业务建议文档 + AI智能洞察 ---")
    # 读取历史指标
    try:
        overall = pd.read_csv(f"{REPORT_DIR}/overall_metrics.csv")
        conv_rate = overall[overall['指标'] == '点击→购买转化率']['数值'].values[0]
        jump_rate = overall[overall['指标'] == '跳失率']['数值'].values[0]
        conv_rate_str = f"{conv_rate:.2%}"
        jump_rate_str = f"{jump_rate:.2%}"
    except:
        conv_rate_str = "待补充"
        jump_rate_str = "待补充"

    try:
        hour_trend = pd.read_csv(f"{REPORT_DIR}/hour_trend.csv")
        peak_hours = hour_trend.nlargest(3, 'PV')['hour'].tolist()
        peak_hours_str = "、".join([f"{h}点" for h in peak_hours])
        peak_hours_detail = "、".join([f"{h}:00-{h + 1}:00" for h in peak_hours])
    except:
        peak_hours_str = "20点、21点、22点"
        peak_hours_detail = "20:00-23:00"

    # 核心用户数据
    high_value_row = segment_stats.iloc[0] if not segment_stats.empty else None
    if high_value_row is not None:
        high_value_cnt = high_value_row['用户数']
        high_value_pct = high_value_row['用户占比']
        high_value_buy_pct = high_value_row['购买贡献占比']
    else:
        high_value_cnt = high_value_pct = high_value_buy_pct = 0

    # AI自动分析文本
    ai_insight = ai_auto_analysis(segment_stats, conv_rate_str, peak_hours_str)

    # 拼接完整报告
    business_md = f"""# 电商用户行为分析 - 业务运营建议
## 一、核心发现摘要
- 整体点击→购买转化率：{conv_rate_str}
- 整体用户跳失率：{jump_rate_str}
- 核心高价值用户规模：{high_value_cnt:,}人，占全量用户{high_value_pct:.1%}，贡献全平台{high_value_buy_pct:.1%}购买量
- 用户活跃高峰时段：{peak_hours_str}（{peak_hours_detail}）
{ai_insight}
## 二、RFM 8分层精细化运营策略
覆盖：重要价值/重要挽留/重要深耕/重要唤回/一般价值/一般挽留/一般潜力/低价值流失 用户
- 重要价值用户：核心私域维护，专属权益+新品优先
- 重要挽留用户：近期活跃度下降，定向优惠券唤醒
- 重要深耕用户：有购买力但频次低，推送关联商品提升复购
- 流失类用户：大促节点低成本批量触达

## 三、商品侧优化建议
### 1. 高点击低转化问题商品优化
详见 `outputs/high_clicks_low_conversion.csv`
- 优化主图、详情页核心卖点
- 校验定价与同品类竞品差距
- 增加运费险、销量背书提升信任

### 2. 热销品类深度运营
详见 `outputs/top_categories.csv`
- 加大流量投放、搭建品类关联推荐
## 四、时间维度运营策略
高峰时段集中推送活动、优惠券、直播预告
## 五、落地A/B测试方案
1. RFM分层差异化推送测试
2. 问题商品详情页改版测试
3. 不同时段推送效果对照测试
---
*本报告基于淘宝用户行为数据清洗分析，含AI智能洞察模块*
"""
    with open(f"{DOCS_DIR}/business_recommendations.md", "w", encoding="utf-8-sig") as f:
        f.write(business_md)
    print("✅ 含AI洞察的业务报告已保存至 docs/")


# ====================== 主程序入口 ======================
if __name__ == "__main__":
    # 执行全流程
    df = load_and_preprocess()
    rfm, seg_stats = rfm_analysis(df)
    cat_buy, prob_item = category_item_analysis(df)
    draw_all_plots(seg_stats, rfm, cat_buy, prob_item)
    generate_business_report(seg_stats)

    print("\n🎉 进阶版项目全部完成！")
    print("📁 输出目录：")
    print("  - 分析结果：outputs/")
    print("  - 业务报告：docs/")
    print("  - 可视化图表：report/")

✅ 数据加载完成，共 98,914,484 行
   时间范围：2017-11-25 00:00:00 至 2017-12-03 23:52:41

--- 1. RFM 8象限 用户分层（行业标准版） ---
   RFM参考日期：2017-12-04
✅ RFM8象限分层结果已保存

--- 2. 商品/品类价值分析 ---
✅ 热门品类、问题商品清单已保存

--- 3. 高级可视化绘图中 ---


C:\Users\AjiaoxuwoLz\AppData\Local\Temp\ipykernel_23428\2001056486.py:170: UserWarning: Glyph 30005 (\N{CJK UNIFIED IDEOGRAPH-7535}) missing from font(s) Arial.
  plt.savefig(f"{IMG_DIR}/user_segment_pie.png", dpi=DPI, bbox_inches='tight')
C:\Users\AjiaoxuwoLz\AppData\Local\Temp\ipykernel_23428\2001056486.py:170: UserWarning: Glyph 21830 (\N{CJK UNIFIED IDEOGRAPH-5546}) missing from font(s) Arial.
  plt.savefig(f"{IMG_DIR}/user_segment_pie.png", dpi=DPI, bbox_inches='tight')
C:\Users\AjiaoxuwoLz\AppData\Local\Temp\ipykernel_23428\2001056486.py:170: UserWarning: Glyph 29992 (\N{CJK UNIFIED IDEOGRAPH-7528}) missing from font(s) Arial.
  plt.savefig(f"{IMG_DIR}/user_segment_pie.png", dpi=DPI, bbox_inches='tight')
C:\Users\AjiaoxuwoLz\AppData\Local\Temp\ipykernel_23428\2001056486.py:170: UserWarning: Glyph 25143 (\N{CJK UNIFIED IDEOGRAPH-6237}) missing from font(s) Arial.
  plt.savefig(f"{IMG_DIR}/user_segment_pie.png", dpi=DPI, bbox_inches='tight')
C:\Users\AjiaoxuwoLz\AppData\Local\Temp\

✅ 全部高级可视化图表已保存至 report/

--- 4. 生成业务建议文档 + AI智能洞察 ---
✅ 含AI洞察的业务报告已保存至 docs/

🎉 进阶版项目全部完成！
📁 输出目录：
  - 分析结果：outputs/
  - 业务报告：docs/
  - 可视化图表：report/
